# Spatial Experiment — Target Detection via Score Matching

**Section 5**: CF-Attn + NeighborMLP vs THANTD/GMM on Pavia-U.

Workflow:
1. Clone/pull repo from GitHub
2. Mount Drive (data file + results only)
3. Dry-run → full run
4. Display figures inline

In [ ]:
# ── Cell 1: Clone repo + install deps ────────────────────────────────────
import os, sys

REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} pull
else:
    !git clone {REPO_URL} {LOCAL_PROJECT}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml

for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())

In [ ]:
# ── Cell 2: Mount Drive (data + results only) ─────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/proj/data', exist_ok=True)

DATA_DST = '/content/proj/data/pavia-u.mat'
if not os.path.exists(DATA_DST):
    for src in [
        '/content/drive/MyDrive/final_paper/pavia-u.mat',
        '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat',
    ]:
        if os.path.exists(src):
            os.symlink(src, DATA_DST)
            print('Data linked:', src)
            break
    else:
        raise FileNotFoundError('Upload pavia-u.mat to /MyDrive/final_paper/')
else:
    print('Data already present.')

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results dir:', RESULTS_DIR)

In [ ]:
# ── Cell 3a: DRY RUN — verify all outputs before full run ─────────────────
RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

!python -u experiments/spatial/run_colab.py \
    --config experiments/spatial/colab.yaml \
    --results_dir {RESULTS_DIR} \
    --dry-run \
    --no-thantd

In [ ]:
# ── Cell 3b: FULL RUN (~2-3 h on T4) ─────────────────────────────────────
# Checkpoints saved after each model — safe to restart on disconnect.
RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

!python -u experiments/spatial/run_colab.py \
    --config experiments/spatial/colab.yaml \
    --results_dir {RESULTS_DIR}

In [ ]:
# ── Cell 4: Display figures inline ───────────────────────────────────────
import os, glob, json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

runs = sorted(glob.glob(os.path.join(RESULTS_DIR, 'colab_*')))
assert runs, 'No results found. Run Cell 3 first.'
run_dir = runs[-1]
print(f'Showing: {run_dir}')

# AUC summary
mpath = os.path.join(run_dir, 'all_metrics.json')
if os.path.exists(mpath):
    all_metrics = json.load(open(mpath))
    print('\n=== AUC Summary (additive) ===')
    for sid_key, sid_data in all_metrics.items():
        for bkey, bdata in sid_data.items():
            if not bkey.startswith('n'): continue
            auc = bdata.get('additive', {}).get('auc', {})
            vals = '  '.join(f'{k}={v:.3f}' for k,v in auc.items() if isinstance(v, float))
            print(f'  {sid_key} {bkey}: {vals}')

# Display aggregated figures
!apt-get install -y -q poppler-utils
for pdf in sorted(glob.glob(os.path.join(run_dir, 'figures', '*.pdf'))):
    png = pdf.replace('.pdf', '.png')
    os.system(f'pdftoppm -r 150 -png -singlefile "{pdf}" "{pdf[:-4]}"')
    if os.path.exists(png):
        img = mpimg.imread(png)
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(img); ax.axis('off')
        ax.set_title(os.path.basename(pdf), fontsize=10)
        plt.tight_layout(); plt.show()